## progan实验

## 实验介绍

生成对抗网络（GAN）是图像生成领域的代表性深度生成模型，其中 ProGAN（Progressive Growing GAN） 通过"由粗到细"的渐进式增长策略，从 4×4 逐级倍增至 128×128，有效稳定了高分辨率人脸生成训练。
## 预期结果

预训练模型本身就能生成正常人脸；微调目标是让部分颜色/构图/纹理更接近 AIGIBench 分布。**小幅但可观察的域适配即为成功**，而非指望 1000 张图重新教会模型人脸结构。



## 第一步：环境与资产准备

首先确认运行环境与项目资产就位，避免后续步骤中途失败。本步一次性完成三件事：

1. **导入依赖与路径设置**：复用 `src/_common.py` 中的设备选择、模型加载、固定潜变量等工具，避免重复代码；
2. **运行环境检查**：确认 Python  / CANN 是否就位（本地无 NPU 时自动回退 CPU，便于先做烟测）；
3. **项目资产检查**：确认 GAN Zoo 源码、预训练权重、训练数据、固定潜变量是否就位。

> notebook 位于项目根目录，`src/` 就在同级。

### 下载资源

训练数据集 `aigibench_128`（998 张 128×128 人脸 PNG，约 29 MB）。下方单元格在首次运行时会从指定网址下载 `aigibench_128.zip` 并解压到 `src/data/aigibench_128/`；若本地已存在则跳过。下载运行所需的源码

In [ ]:
import subprocess
from pathlib import Path

GAN_ZOO_URL = "https://github.com/facebookresearch/pytorch_GAN_zoo.git"
GAN_ZOO_DIR = Path.cwd() / "src" / "code" / "pytorch_GAN_zoo"

GAN_ZOO_DIR.parent.mkdir(parents=True, exist_ok=True)

if GAN_ZOO_DIR.exists():
    print(f"pytorch_GAN_zoo 已存在，跳过下载: {GAN_ZOO_DIR}")
else:
    subprocess.run(["git", "clone", GAN_ZOO_URL, str(GAN_ZOO_DIR)], check=True)
    print(f"pytorch_GAN_zoo 下载完成: {GAN_ZOO_DIR}")


pgan_file = GAN_ZOO_DIR / "models" / "progressive_gan.py"

if not pgan_file.exists():
    raise FileNotFoundError(f"未找到 ProGAN 源文件: {pgan_file}")

code = pgan_file.read_text(encoding="utf-8")

old = "betas=[0, 0.99]"
new = "betas=(0.0, 0.99)"

if old in code:
    code = code.replace(old, new)
    pgan_file.write_text(code, encoding="utf-8")
    print("已应用 PyTorch 2.x Adam betas 兼容补丁")
else:
    print("Adam betas 兼容补丁已存在，无需重复修改")


import os, shutil, subprocess, tempfile
from pathlib import Path


In [ ]:
# 从 GitCode 下载 AIGIBench 数据集到 src/data，并自动解压
import os
import shutil
import subprocess
import tempfile
import zipfile
from pathlib import Path

DATA_REPO_URL = "https://gitcode.com/qq_627548571/face.git"
DATA_REPO_BRANCH = "main"
REMOTE_DATA_PATH = "experiment7/data"

# notebook 通常位于项目根目录；兼容从子目录启动的情况
NB_DIR = Path.cwd()
SRC_DIR = NB_DIR / "src"
if not SRC_DIR.exists() and (NB_DIR.parent / "src").exists():
    SRC_DIR = NB_DIR.parent / "src"

LOCAL_DATA_ROOT = SRC_DIR / "data"
EXPECTED_DATA_DIR = LOCAL_DATA_ROOT / "aigibench_128"
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp"}


def _image_count(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(
        1 for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
    )


def _run(cmd, *, cwd=None, env=None):
    print("+", " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)


def _is_lfs_pointer(path: Path) -> bool:
    try:
        with path.open("rb") as f:
            head = f.read(200)
        return head.startswith(b"version https://git-lfs.github.com/spec/v1")
    except OSError:
        return False


def _ensure_git_lfs():
    """仅当数据文件确实是 Git LFS 指针时才安装/启用 git-lfs。"""
    has_lfs = subprocess.run(
        ["git", "lfs", "version"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    ).returncode == 0

    if not has_lfs:
        apt = shutil.which("apt-get")
        if apt:
            if hasattr(os, "geteuid") and os.geteuid() == 0:
                prefix = []
            elif shutil.which("sudo"):
                prefix = ["sudo"]
            else:
                raise RuntimeError(
                    "数据文件使用 Git LFS，但当前环境未安装 git-lfs，"
                    "且没有 sudo/root 权限自动安装。请先安装 git-lfs 后重试。"
                )
            _run(prefix + [apt, "update"])
            _run(prefix + [apt, "install", "-y", "git-lfs"])
        else:
            raise RuntimeError(
                "数据文件使用 Git LFS，但当前环境未安装 git-lfs。"
                "请先安装 git-lfs 后重试。"
            )

    _run(["git", "lfs", "install", "--skip-repo"])


existing_count = _image_count(EXPECTED_DATA_DIR)

if existing_count >= 100:
    print(
        f"AIGIBench 数据集已存在，跳过下载: "
        f"{EXPECTED_DATA_DIR} ({existing_count} 张图片)"
    )
else:
    if shutil.which("git") is None:
        raise RuntimeError("未检测到 git，请先安装 Git。")

    # 临时仓库放在 src 下；完成后会自动删除。
    tmp_repo = Path(tempfile.mkdtemp(prefix=".face_data_", dir=SRC_DIR))

    try:
        # 跳过 LFS 自动下载，先只拿 Git 元数据和目标目录，避免误拉取仓库中的大模型。
        env = os.environ.copy()
        env["GIT_LFS_SKIP_SMUDGE"] = "1"

        _run([
            "git", "clone",
            "--depth", "1",
            "--branch", DATA_REPO_BRANCH,
            "--no-checkout",
            DATA_REPO_URL,
            str(tmp_repo),
        ], env=env)

        _run([
            "git", "-C", str(tmp_repo),
            "sparse-checkout", "init", "--cone",
        ])

        _run([
            "git", "-C", str(tmp_repo),
            "sparse-checkout", "set", REMOTE_DATA_PATH,
        ])

        _run([
            "git", "-C", str(tmp_repo),
            "checkout", DATA_REPO_BRANCH,
        ], env=env)

        remote_data_dir = tmp_repo / REMOTE_DATA_PATH
        if not remote_data_dir.exists():
            raise FileNotFoundError(
                f"仓库中未找到数据目录: {REMOTE_DATA_PATH}"
            )

        # 优先寻找 notebook 描述中对应的 aigibench_128.zip；
        # 若仓库中只有一个 zip，也兼容自动选取。
        source_zip = remote_data_dir / "aigibench_128.zip"
        if not source_zip.exists():
            zip_candidates = sorted(remote_data_dir.rglob("*.zip"))
            if len(zip_candidates) == 1:
                source_zip = zip_candidates[0]
            elif len(zip_candidates) == 0:
                source_dir = remote_data_dir / "aigibench_128"
                if source_dir.exists():
                    if EXPECTED_DATA_DIR.exists():
                        shutil.rmtree(EXPECTED_DATA_DIR)
                    shutil.copytree(source_dir, EXPECTED_DATA_DIR)
                    source_zip = None
                else:
                    raise FileNotFoundError(
                        f"{REMOTE_DATA_PATH} 中未找到 aigibench_128.zip "
                        "或 aigibench_128 数据目录。"
                    )
            else:
                names = ", ".join(p.name for p in zip_candidates)
                raise RuntimeError(
                    "数据目录中存在多个 zip，无法确定应解压哪一个: " + names
                )

        if source_zip is not None:
            # 如果 zip 是 Git LFS 指针，只拉取 experiment7/data 下的 LFS 对象。
            if _is_lfs_pointer(source_zip):
                print("检测到数据压缩包使用 Git LFS，正在拉取实际文件...")
                _ensure_git_lfs()
                _run([
                    "git", "-C", str(tmp_repo),
                    "lfs", "pull",
                    "--include", f"{REMOTE_DATA_PATH}/**",
                ])

            if not zipfile.is_zipfile(source_zip):
                raise RuntimeError(
                    f"下载得到的文件不是有效 ZIP: {source_zip}\n"
                    "若该文件使用 Git LFS，请检查 git-lfs 是否成功安装并拉取。"
                )

            # 按要求将压缩包保存在 src/data 下，再从这里解压。
            local_zip = LOCAL_DATA_ROOT / source_zip.name
            shutil.copy2(source_zip, local_zip)
            print(f"数据压缩包已下载到: {local_zip}")

            with zipfile.ZipFile(local_zip, "r") as zf:
                zf.extractall(LOCAL_DATA_ROOT)
            print(f"数据集已解压到: {LOCAL_DATA_ROOT}")

        # 兼容压缩包额外套了一层目录的情况。
        if not EXPECTED_DATA_DIR.exists():
            matches = [
                p for p in LOCAL_DATA_ROOT.rglob("aigibench_128")
                if p.is_dir() and p != EXPECTED_DATA_DIR
            ]
            if len(matches) == 1:
                shutil.move(str(matches[0]), str(EXPECTED_DATA_DIR))

        final_count = _image_count(EXPECTED_DATA_DIR)
        if final_count < 100:
            raise RuntimeError(
                f"数据集解压后图片数量异常: {final_count} 张；"
                f"请检查目录 {EXPECTED_DATA_DIR}"
            )

        print(
            f"AIGIBench 数据集准备完成: "
            f"{EXPECTED_DATA_DIR} ({final_count} 张图片)"
        )

    finally:
        shutil.rmtree(tmp_repo, ignore_errors=True)

In [ ]:
import sys, subprocess, os, platform
from pathlib import Path

NB_DIR = Path.cwd()
SCRIPTS = NB_DIR / "src"
if not SCRIPTS.exists():
    SCRIPTS = NB_DIR.parent / "src"
sys.path.insert(0, str(SCRIPTS))

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as IPImage, Markdown, clear_output
%matplotlib inline

import _common
from _common import (
    PROJECT_DIR, REPO_DIR, DATA_DIR, CKPT_PATH, OUTPUT_DIR, FIXED_Z_PATH,
    setup_device, load_pgans, get_or_create_fixed_z, generate_grid, save_checkpoint,
    DEFAULT_LR, DEFAULT_NROW, log,
)

print("项目根 PROJECT_DIR =", PROJECT_DIR)
print("PyTorch            =", torch.__version__)
import importlib, datetime

def ver(modname):
    try:
        m = importlib.import_module(modname)
        return getattr(m, "__version__", "?")
    except Exception as e:
        return f"未安装 ({type(e).__name__})"

rows = [
    ("Python", platform.python_version()),
    ("PyTorch", torch.__version__),
    ("torchvision", ver("torchvision")),
    ("torch_npu", ver("torch_npu")),
    ("Pillow", ver("PIL")),
    ("numpy", np.__version__),
    ("matplotlib", ver("matplotlib")),
    ("平台", platform.platform()),
]
for k, v in rows:
    print(f"  {k:<14}: {v}")

# CANN 环境变量
print("\nCANN 相关环境变量:")
cann_keys = [k for k in os.environ if "ASCEND" in k or "CANN" in k]
for k in cann_keys:
    print(f"  {k} = {os.environ[k]}")
if not cann_keys:
    print("  (未检测到 ASCEND/CANN 环境变量；若在 NPU 上请先 source set_env.sh)")
dev = setup_device()
mode, device = dev["mode"], dev["device"]
print(f"运行设备: {mode}  ->  {device}")

# 设备自检：矩阵乘法 + 计时
import time
x = torch.randn(2048, 2048, device=device)
t0 = time.time(); y = (x @ x).sum().item(); dt = time.time() - t0
print(f"2048×2048 矩阵乘法自检: sum={y:.2f}, 耗时 {dt*1000:.1f} ms  OK")

# 设备信息表
info = {"mode": mode, "device": str(device), "useGPU": dev["useGPU"]}
if mode == "npu":
    info["npu_count"] = torch.npu.device_count()
    try: info["npu_name"] = torch.npu.get_device_name(0)
    except Exception: info["npu_name"] = "?"
elif mode == "cuda":
    info["cuda_count"] = torch.cuda.device_count()
    info["cuda_name"] = torch.cuda.get_device_name(0)
print("\n设备信息:")
for k, v in info.items(): print(f"  {k}: {v}")
def human_size(p):
    try: s = p.stat().st_size
    except OSError: return "不存在"
    for u in ["B","KB","MB","GB"]:
        if s < 1024: return f"{s:.1f} {u}"
        s /= 1024
    return f"{s:.1f} TB"

checks = [
    ("预训练权重", CKPT_PATH, CKPT_PATH.exists()),
    ("训练数据目录", DATA_DIR, DATA_DIR.exists()),
    ("固定潜变量", FIXED_Z_PATH, FIXED_Z_PATH.exists()),
]
print(f"{'资产':<16}{'路径':<60}{'大小':<12}{'状态'}")
print("-"*100)
for name, p, ok in checks:
    print(f"{name:<16}{str(p)[-58:]:<60}{human_size(p):<12}{'✅' if ok else '❌'}")

# 数据图像数量
if DATA_DIR.exists():
    faces = sorted(p for p in DATA_DIR.iterdir() if p.suffix.lower() in {".png",".jpg",".jpeg",".webp"})
    print(f"\n训练图像数量: {len(faces)} 张 (期望约 998)")
else:
    faces = []
    print("\n⚠ 训练数据目录不存在！")

assert len(faces) >= 100, "训练数据不足，无法继续"

## 第二步：数据可视化与统计

**数据处理流程**：原图 → RGB → 中心正方形裁剪 → 缩放 128×128 → PNG → 训练时随机水平翻转 → 归一化到 [-1,1]

下面展示 64 张样本，并统计像素分布。这一步对判断数据域、归一化是否合理很有帮助——学员应观察样本的风格、肤色、构图分布，与预训练模型（CelebA）的差异正是微调要弥合的"域间距"。

In [ ]:
# 8×8 样本网格
rng = np.random.default_rng(20260710)
sample = rng.choice(faces, size=min(64, len(faces)), replace=False)
fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for ax, p in zip(axes.ravel(), sample):
    ax.imshow(Image.open(p)); ax.axis("off")
for ax in axes.ravel()[len(sample):]: ax.axis("off")
plt.suptitle("AIGIBench 人脸样本 (128×128)", fontsize=16, y=0.92)
plt.tight_layout(); plt.show()

In [ ]:
# 像素统计：随机抽 256 张计算 RGB 通道均值/标准差 + 直方图
stat_sample = rng.choice(faces, size=min(256, len(faces)), replace=False)
all_pixels = []
for p in stat_sample:
    arr = np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 255.0
    all_pixels.append(arr.reshape(-1, 3))
all_pixels = np.concatenate(all_pixels, axis=0)

mean = all_pixels.mean(axis=0)
std = all_pixels.std(axis=0)
print(f"采样 {len(stat_sample)} 张，共 {len(all_pixels)} 个像素")
print(f"RGB 均值 = {mean}  (归一化到[-1,1]后 ≈ {mean*2-1})")
print(f"RGB 标准差 = {std}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
channels = ["R", "G", "B"]
colors = ["tab:red", "tab:green", "tab:blue"]
for c, ax, col in zip(range(3), axes, colors):
    ax.hist(all_pixels[:, c], bins=50, color=col, alpha=0.8)
    ax.set_title(f"{channels[c]} 通道像素分布 (0-1)")
    ax.set_xlabel("像素值"); ax.set_ylabel("频数")
plt.suptitle("AIGIBench 人脸像素分布统计", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# 单张细节查看（原图 vs 归一化后）
p = sample[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
orig = np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 255.0
axes[0].imshow(orig); axes[0].set_title("原始 (0-1)"); axes[0].axis("off")
norm = orig * 2 - 1  # 归一化到 [-1,1]
axes[1].imshow((norm + 1) / 2); axes[1].set_title("归一化 [-1,1] (训练输入)"); axes[1].axis("off")
plt.suptitle(f"单张细节: {p.name}", fontsize=12)
plt.tight_layout(); plt.show()

## 第三步：加载预训练 ProGAN

从本地 GAN Zoo 仓库构建 PGAN，再加载 `celebaCropped_s5_i83000-2b0acc76.pth`（CelebA 128×128 checkpoint）。

**自动下载**：若本地不存在预训练权重，会先从指定网址下载到 `src/pretrained/` 目录后再加载。下载地址在下方单元格的 `CKPT_URL` 变量中配置（目前为待定占位地址，部署时替换为实际发布链接）。

checkpoint 包含：预训练 Generator、Discriminator、EMA(平均) Generator、分辨率与全部参数结构。

In [ ]:
# GitCode 仓库：只拉取 experiment7/premodel；LFS 大文件由 git lfs pull 获取
import os, shutil, subprocess, tempfile
from pathlib import Path
import torch.nn as nn

REPO_URL = "https://gitcode.com/qq_627548571/face.git"
REPO_BRANCH = "main"
MODEL_REPO_PATH = "experiment7/premodel"

def _run(cmd, *, cwd=None, env=None):
    print("+", " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

def _ensure_git_lfs():
    if shutil.which("git") is None:
        raise RuntimeError("未检测到 git，请先安装 Git。")

    has_lfs = subprocess.run(
        ["git", "lfs", "version"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    ).returncode == 0

    if not has_lfs:
        apt = shutil.which("apt-get")
        if apt:
            if hasattr(os, "geteuid") and os.geteuid() == 0:
                prefix = []
            elif shutil.which("sudo"):
                prefix = ["sudo"]
            else:
                raise RuntimeError(
                    "未检测到 git-lfs，且当前用户无权限自动安装。请先安装 git-lfs 后重试。"
                )
            _run(prefix + [apt, "update"])
            _run(prefix + [apt, "install", "-y", "git-lfs"])
        else:
            raise RuntimeError("未检测到 git-lfs。请先安装 git-lfs 后重试。")

    _run(["git", "lfs", "install", "--skip-repo"])


def fix_progan_npu(model):
    if mode != "npu":
        return model

    # BaseGAN.test()、buildNoiseData() 等都会使用 model.device
    model.device = device

    for name in ("netG", "netD", "avgG"):
        module = getattr(model, name, None)

        if module is None:
            continue

        # 去掉老版 CUDA DataParallel
        if isinstance(module, nn.DataParallel):
            print(f"[progan] unwrap DataParallel: {name}")
            module = module.module

        # 明确移动到 Ascend NPU
        module = module.to(device)
        setattr(model, name, module)

    print(f"[progan] NPU compatibility fixed -> {device}")

    # 打印一下确认网络真实所在设备
    for name in ("netG", "netD", "avgG"):
        module = getattr(model, name, None)
        if module is None:
            continue

        param = next(module.parameters(), None)
        param_device = param.device if param is not None else "no parameters"

        print(
            f"  {name:<4}: "
            f"type={type(module).__name__}, "
            f"device={param_device}"
        )

    return model


# ============================================================
# 若本地不存在，则通过 Git sparse-checkout + Git LFS 拉取
# ============================================================
if not CKPT_PATH.exists():
    _ensure_git_lfs()
    CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
    src_dir = CKPT_PATH.parent.parent
    src_dir.mkdir(parents=True, exist_ok=True)

    tmp_repo = Path(
        tempfile.mkdtemp(prefix=".face_premodel_lfs_", dir=src_dir)
    )

    try:
        env = os.environ.copy()
        env["GIT_LFS_SKIP_SMUDGE"] = "1"

        _run([
            "git", "clone",
            "--depth", "1",
            "--branch", REPO_BRANCH,
            "--no-checkout",
            REPO_URL,
            str(tmp_repo)
        ], env=env)

        _run([
            "git", "-C", str(tmp_repo),
            "sparse-checkout", "init", "--cone"
        ])

        _run([
            "git", "-C", str(tmp_repo),
            "sparse-checkout", "set", MODEL_REPO_PATH
        ])

        _run([
            "git", "-C", str(tmp_repo),
            "checkout", REPO_BRANCH
        ], env=env)

        _run([
            "git", "-C", str(tmp_repo),
            "lfs", "pull",
            "--include", f"{MODEL_REPO_PATH}/**"
        ])

        remote_model_dir = tmp_repo / MODEL_REPO_PATH

        matches = (
            list(remote_model_dir.rglob(CKPT_PATH.name))
            if remote_model_dir.exists()
            else []
        )

        if not matches:
            raise FileNotFoundError(
                f"仓库 {MODEL_REPO_PATH} 中未找到预训练权重: "
                f"{CKPT_PATH.name}"
            )

        shutil.copy2(matches[0], CKPT_PATH)

        print(
            f"Git LFS 拉取完成: "
            f"{CKPT_PATH}  ({human_size(CKPT_PATH)})"
        )

    finally:
        shutil.rmtree(tmp_repo, ignore_errors=True)

else:
    print(
        f"预训练权重已存在: "
        f"{CKPT_PATH}  ({human_size(CKPT_PATH)})"
    )


# ============================================================
# 加载预训练 ProGAN
# ============================================================
model = load_pgans(
    base_lr=DEFAULT_LR,
    useGPU=dev["useGPU"]
)

model = fix_progan_npu(model)


print("模型加载完成")
print(f"  分辨率输出: {model.config.dimOutput}")
print(f"  潜变量维度: {model.config.noiseVectorDim}")
print(f"  学习率    : {model.config.learningRate}")
print(f"  损失模式  : {model.config.lossCriterion}")


### 生成器网络结构与参数量

ProGAN 采用渐进式增长（progressive growing），生成器由多个分辨率层级组成。

In [ ]:
G = model.getOriginalG()
print(G)

# 参数量统计
def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

g_total, g_train = count_params(G)
d_total, d_train = count_params(model.getOriginalD())
avg_total, _ = count_params(model.getOriginalAvgG())
print(f"\n生成器(G)   参数量: {g_total:,}  (可训练 {g_train:,})")
print(f"判别器(D)   参数量: {d_total:,}  (可训练 {d_train:,})")
print(f"平均生成器  参数量: {avg_total:,}")
print(f"模型总参数量: {g_total + d_total + avg_total:,}")

## 第四步：基线生成（微调前）

固定 16 个潜变量（512 维），保存到 `src/fixed_latent/fixed_z.pt`。
这 16 个潜变量将在整个微调过程中**保持不变**，用于直观对比模型如何逐步适应 AIGIBench 分布。这是 GAN 微调实验的关键技巧——固定输入才能把生成差异归因于模型权重变化。

In [ ]:
fixed_z = get_or_create_fixed_z(model, n=16)
print("fixed_z shape:", tuple(fixed_z.shape))

baseline_path = OUTPUT_DIR / "baseline" / "baseline_grid.png"
generate_grid(model, fixed_z, baseline_path, nrow=DEFAULT_NROW)
display(IPImage(str(baseline_path)))
Markdown("↑ **微调前基线**：预训练 ProGAN 在 16 个固定潜变量下的生成结果。")

In [ ]:
# 基线单张放大查看（第 1 张）
with torch.no_grad():
    model.netG.eval()
    one = model.test(fixed_z[:1], getAvG=True, toCPU=True).clamp(-1, 1)
one_img = ((one[0].permute(1,2,0).numpy() + 1) / 2)
plt.figure(figsize=(4, 4))
plt.imshow(one_img); plt.axis("off")
plt.title("基线单张放大 (潜变量 #0)"); plt.show()

## 第五步：用 AIGIBench 微调

**微调参数**（已调整为快速配置）：

- **resolution = 128**：输出分辨率
- **batch_size = 8**：每步批大小，加大可加速（OOM 就退回 4）
- **learning_rate = 0.00001**：微调学习率，取较小值以避免破坏预训练权重
- **total_steps = 100**：微调步数，短程域适配，节省时间
- **save_every / sample_every = 50**：每 50 步存 checkpoint + 网格

下方单元格执行训练循环，**实时绘制 loss 曲线**并在每 50 步展示固定潜变量的生成网格。
想更彻底可把 `TOTAL_STEPS` 改为 200/500；想更快可改为 50。

> 保存的 checkpoint：`step_0000`(微调前) → `step_0050` → `step_0100`

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms
import csv

# ===== 可调参数（快速配置）=====
TOTAL_STEPS   = 100       # 想更彻底可改 200/500，想更快改 50
BATCH_SIZE    = 8         # OOM 就退回 4
LR            = 1e-5
SAVE_EVERY    = 50
SAMPLE_EVERY  = 50
NUM_WORKERS   = 4
# ================================

class FaceDataset(torch.utils.data.Dataset):
    def __init__(self, root):
        self.paths = sorted(p for p in root.iterdir() if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"})
        self.transform = transforms.Compose([
            transforms.Resize(128), transforms.CenterCrop(128),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            return self.transform(im.convert("RGB"))

dataset = FaceDataset(DATA_DIR)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS)
print(f"dataset = {len(dataset)} images, batch={BATCH_SIZE}, total_steps={TOTAL_STEPS}")
print(f"预计每步处理 {BATCH_SIZE} 张，共 {TOTAL_STEPS} 步 ≈ {TOTAL_STEPS*BATCH_SIZE} 张图次")

In [ ]:
# 训练循环 + 实时可视化
loss_log = []
step = 0
model.netG.train(); model.netD.train()

while step < TOTAL_STEPS:
    for real_images in loader:
        losses = model.optimizeParameters(real_images)
        step += 1
        loss_log.append({
            "step": step,
            "lossD_real": float(losses.get("lossD_real", float("nan"))),
            "lossD_fake": float(losses.get("lossD_fake", float("nan"))),
            "lossG":      float(losses.get("lossG", float("nan"))),
        })

        if step % SAMPLE_EVERY == 0 or step >= TOTAL_STEPS:
            out_dir = OUTPUT_DIR / f"step_{step:04d}"
            grid_path = out_dir / "grid.png"
            generate_grid(model, fixed_z, grid_path, nrow=DEFAULT_NROW)
            save_checkpoint(model, out_dir / "progan_finetuned.pth")

            clear_output(wait=True)
            fig, (ax_loss, ax_img) = plt.subplots(1, 2, figsize=(13, 5))
            s = [r["step"] for r in loss_log]
            for k, c in [("lossD_real","tab:blue"),("lossD_fake","tab:orange"),("lossG","tab:green")]:
                ax_loss.plot(s, [r[k] for r in loss_log], label=k, color=c)
            ax_loss.set_xlabel("step"); ax_loss.set_ylabel("loss"); ax_loss.legend(); ax_loss.grid(True, alpha=0.3)
            ax_loss.set_title(f"loss @ step {step}/{TOTAL_STEPS}")
            ax_img.imshow(Image.open(grid_path)); ax_img.axis("off"); ax_img.set_title(f"step {step} 固定潜变量生成")
            plt.show()
            print(f"step={step}/{TOTAL_STEPS}  {losses}", flush=True)
            model.netG.train(); model.netD.train()

        if step % 20 == 0 and step % SAMPLE_EVERY != 0:
            print(f"step={step}/{TOTAL_STEPS}  {losses}", flush=True)

        if step >= TOTAL_STEPS:
            break

# 写 loss 日志
loss_csv = OUTPUT_DIR / "loss_log.csv"
with loss_csv.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["step","lossD_real","lossD_fake","lossG"])
    w.writeheader(); w.writerows(loss_log)
print("微调完成，loss 日志 ->", loss_csv)

## 第六步：提前停止（人工判断）

微调不是越久越好。若出现以下现象应立即停止并回退到之前的 checkpoint：

- 五官开始扭曲
- 所有脸越来越相似（模式坍缩）
- 颜色突然异常
- 背景出现重复纹理

最后的最好模型很可能是 `step_0050` 或 `step_0100`，**不一定是最后一步**。请结合上方各 step 的网格图人工判断，这是 GAN 微调区别于判别式模型训练的重要环节。

## 第七步：结果汇总对比

调用 `compare_results.py` 把 baseline 与各 step 的网格横向拼接，并绘制 loss 曲线，便于整体回顾微调过程。

In [ ]:
res = subprocess.run([sys.executable, str(SCRIPTS / "compare_results.py")], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:", res.stderr)

cmp_dir = OUTPUT_DIR / "comparison"
for name, fn in [("对比网格 (baseline → step_0250 → 0500 → ...)","compare_grid.png"),
                 ("loss 曲线","loss_curve.png")]:
    p = cmp_dir / fn
    if p.exists():
        print(f"=== {name} ===")
        display(IPImage(str(p)))

## 第八步：微调前后随机生成对比

用**同一组新的随机潜变量**（非固定那 16 个）分别送入微调前(预训练)和微调后(最佳 checkpoint)的模型，并排对比，直观看到微调对生成分布的影响。固定随机种子保证两边输入完全一致，差异仅来自模型权重。

In [ ]:
# 找最佳可用 finetuned checkpoint（优先 step_0100）
candidates = [100, 50, 250, 500]

best_ckpt = None
best_step = None

for s in candidates:
    p = OUTPUT_DIR / f"step_{s:04d}" / "progan_finetuned.pth"
    if p.exists():
        best_ckpt = p
        best_step = s
        break


if best_ckpt is None:
    print("未找到任何微调 checkpoint，跳过本步。请先运行第五步微调。")

else:
    print(
        f"最佳 checkpoint: {best_ckpt}  "
        f"(step_{best_step:04d})"
    )

    # ========================================================
    # 8 个新的随机潜变量
    # 固定种子，保证微调前 / 微调后使用完全相同的 latent
    # ========================================================
    rng_z = torch.Generator().manual_seed(42)

    rand_z = torch.randn(
        8,
        model.config.noiseVectorDim,
        generator=rng_z
    )


    # ========================================================
    # 微调前：重新加载原始预训练模型
    # ========================================================
    print("\n加载预训练模型（微调前）...")

    model_pre = load_pgans(
        base_lr=DEFAULT_LR,
        useGPU=dev["useGPU"]
    )

    model_pre = fix_progan_npu(model_pre)


    # ========================================================
    # 微调后：重新建立模型并加载微调 checkpoint
    # ========================================================
    print("\n加载微调后模型...")

    model_post = load_pgans(
        base_lr=DEFAULT_LR,
        useGPU=dev["useGPU"]
    )
    model_post = fix_progan_npu(model_post)

    state = torch.load(
        best_ckpt,
        map_location="cpu",
        weights_only=False
    )

    model_post.load_state_dict(state)

    model_post = fix_progan_npu(model_post)


    model_pre.netG.eval()
    model_post.netG.eval()

    if hasattr(model_pre, "avgG") and model_pre.avgG is not None:
        model_pre.avgG.eval()

    if hasattr(model_post, "avgG") and model_post.avgG is not None:
        model_post.avgG.eval()


    with torch.no_grad():

        img_pre = model_pre.test(
            rand_z,
            getAvG=True,
            toCPU=True
        ).clamp(-1, 1)

        img_post = model_post.test(
            rand_z,
            getAvG=True,
            toCPU=True
        ).clamp(-1, 1)

    fig, axes = plt.subplots(
        2,
        8,
        figsize=(20, 5)
    )

    for i in range(8):

        pre_img = (
            img_pre[i]
            .permute(1, 2, 0)
            .numpy()
            + 1
        ) / 2

        post_img = (
            img_post[i]
            .permute(1, 2, 0)
            .numpy()
            + 1
        ) / 2

        axes[0, i].imshow(pre_img)
        axes[0, i].axis("off")

        axes[1, i].imshow(post_img)
        axes[1, i].axis("off")


    axes[0, 0].set_ylabel(
        "Before fine-tuning",
        fontsize=13
    )

    axes[1, 0].set_ylabel(
        f"After fine-tuning\n(step_{best_step:04d})",
        fontsize=13
    )

    plt.suptitle(
        "Same Latent Vectors: Before vs After Fine-tuning",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()

    del model_pre, model_post

    if mode == "cuda":
        torch.cuda.empty_cache()

    elif mode == "npu":
        try:
            torch.npu.empty_cache()
        except Exception:
            pass


## 实验结论

本实验在 Ascend NPU 上运行 ProGAN，加载公开预训练权重，并使用约 1000 张 AIGIBench 人脸图像进行目标域微调。
实验通过固定潜变量比较微调前后生成结果，观察微调步数对人脸结构、纹理特征及模型稳定性的影响。

**预期结论**：
- 微调前：已能生成可见人脸（基线保障）
- 微调后：部分颜色、构图或纹理更接近 AIGIBench，小幅但可观察的域适配即为成功
